
### Test Coverage:
1. **Fact Volume Test**: Ensures the core `fact_listings` table is fully populated.
2. **SCD Type 2 Audit**: Verifies that dimension tables correctly implemented historical tracking metadata (`__START_AT`, `__END_AT`).
3. **Aggregate Availability**: Confirms all materialized views/aggregate tables are populated.
4. **KPI Business Logic**: Validates that minimum data points exist for trend reporting.
5. **Star Schema Integrity**: Checks the join quality between the Fact table and Location Dimension to ensure minimal orphaned records.

# Setup & Dynamic Configuration

In [0]:
# =========================================================
# SETUP: WIDGETS & ENVIRONMENT VARIABLES
# =========================================================
from pyspark.sql import functions as F

# 1. SETUP WIDGETS
dbutils.widgets.text("project_catalog", "vstone_catalog")
dbutils.widgets.text("gold_schema", "gold")

# 2. ASSIGN VARIABLES
CATALOG = dbutils.widgets.get("project_catalog")
SCHEMA = dbutils.widgets.get("gold_schema")
GOLD = f"{CATALOG}.{SCHEMA}"

print(f" INITIALIZING GOLD QA SUITE FOR: {GOLD}")
print("="*60)

# Core Tables & Dimension Testing

In [0]:
# =========================================================
# TEST SUITE 1: FACT & DIMENSION INTEGRITY
# =========================================================

def test_gold_fact_table_populated():
    """Requirement: Gold fact_listings must have significant rows"""
    cnt = spark.table(f"{GOLD}.fact_listings").count()
    # Fact listings typically has ~1.1M rows
    assert cnt > 100000, f"FAIL: fact_listings has only {cnt} rows — too few for the source volume!"
    print(f"    PASS: fact_listings is populated ({cnt:,} rows)")

def test_gold_scd2_columns_on_all_dims():
    """Requirement: Applicable dimensions must have SCD2 columns"""
    # FIX: Removed 'dim_date' as we correctly made it a static sequence without SCD2
    scd2_dims = ["dim_car", "dim_location", "dim_listing_details"]
    
    for dim in scd2_dims:
        df = spark.table(f"{GOLD}.{dim}")
        # DLT Metadata columns check
        assert "__START_AT" in df.columns, f"FAIL: {dim} missing __START_AT!"
        assert "__END_AT" in df.columns, f"FAIL: {dim} missing __END_AT!"
        print(f"    PASS: {dim} has SCD2 metadata (__START_AT, __END_AT)")
        
    # Quick check for static dim_date
    date_cnt = spark.table(f"{GOLD}.dim_date").count()
    assert date_cnt > 0, "FAIL: dim_date is empty!"
    print(f"    PASS: dim_date is populated (Static Calendar)")

# Aggregation & KPI Testing

In [0]:
# =========================================================
# TEST SUITE 2: AGGREGATES & BUSINESS LOGIC
# =========================================================

def test_gold_all_aggregates_populated():
    """Requirement: Verify all Aggregate tables from Catalog"""
    # FIX: Added the 5th aggregate we created for the "Top 10" requirement
    agg_tables = [
        "agg_brand_location_performance",
        "agg_comprehensive_kpi_cube",
        "agg_monthly_sales_trend",
        "agg_regional_market_depth",
        "agg_top_10_brands_by_spend" 
    ]
    for agg in agg_tables:
        cnt = spark.table(f"{GOLD}.{agg}").count()
        assert cnt > 0, f"FAIL: Aggregate {agg} is EMPTY!"
        print(f"    PASS: {agg} is populated ({cnt:,} rows)")

def test_gold_kpi_requirements():
    """Requirement: Check specific KPIs (Monthly Trends & Top Brands)"""
    # 1. Monthly sales trend check
    trend_cnt = spark.table(f"{GOLD}.agg_monthly_sales_trend").count()
    assert trend_cnt >= 12, f"FAIL: Not enough data points for Monthly Trend!"
    
    # 2. Regional depth check (Sales by region/category)
    region_cnt = spark.table(f"{GOLD}.agg_regional_market_depth").count()
    assert region_cnt > 0, "FAIL: Regional market analysis is empty!"
    print(f"    PASS: Monthly and Regional KPI data thresholds met.")

# Star Schema Join Quality Testing

In [0]:
# =========================================================
# TEST SUITE 3: STAR SCHEMA CONNECTIVITY
# =========================================================

def test_gold_geolocation_join_quality():
    """Requirement: Verify join rate between Fact and Location Dimension"""
    fact_cnt = spark.table(f"{GOLD}.fact_listings").count()
    
    # Using specific keys city_prepositional as PK
    joined_cnt = spark.sql(f"""
        SELECT COUNT(*) as c
        FROM {GOLD}.fact_listings f
        JOIN {GOLD}.dim_location d ON f.location_key = d.city_prepositional
        WHERE d.__END_AT IS NULL
    """).collect()[0]['c']
    
    join_rate = round(joined_cnt / fact_cnt * 100, 1)
    
    # 60% threshold requirement
    assert join_rate >= 60.0, f"FAIL: Low join rate {join_rate}% (Min 60% required)!"
    print(f"    PASS: Geolocation join rate = {join_rate}% ({joined_cnt:,}/{fact_cnt:,} matched)")

In [0]:
# =========================================================
# EXECUTION ENGINE
# =========================================================

print("\n EXECUTING PROJECT GOLD LAYER QA TESTS...\n")
print("-" * 60)

try:
    test_gold_fact_table_populated()
    print("-" * 30)
    test_gold_scd2_columns_on_all_dims()
    print("-" * 30)
    test_gold_all_aggregates_populated()
    print("-" * 30)
    test_gold_kpi_requirements()
    print("-" * 30)
    test_gold_geolocation_join_quality()
    print("-" * 60)
    
    print("\n SUCCESS: ALL GOLD LAYER TESTS PASSED! DATA IS DASHBOARD-READY.")

except AssertionError as e:
    print(f"\n TEST FAILED: {str(e)}")
except Exception as e:
    print(f"\n SYSTEM ERROR: {str(e)}")